# Phân tích số liệu dữ liệu bất động sản Zillow

**Mục tiêu:** Khám phá dữ liệu bằng số (numerical exploration) trước khi trực quan hóa, gồm:
1. Tổng quan cấu trúc dữ liệu
2. Thống kê mô tả (Descriptive Statistics)
3. Phân tích dữ liệu khuyết và trùng lặp
4. Bảng thống kê theo loại hình nhà (groupby)
5. Ma trận tương quan với biến mục tiêu `price`


## 0. Import thư viện & đọc dữ liệu

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 150)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

df = pd.read_csv('zillow_final.csv')
print(f"Đã đọc dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột")


## 1. Tổng quan cấu trúc dữ liệu

Kiểm tra số dòng, số cột, kiểu dữ liệu và 5 dòng đầu tiên để hiểu tổng thể bộ dữ liệu.

In [ ]:
print(f"Số dòng: {df.shape[0]}")
print(f"Số cột : {df.shape[1]}")


In [ ]:
df.info()


In [ ]:
df.head()


**Nhận xét:**
- Bộ dữ liệu gồm 4,468 dòng (bất động sản) và 33 cột.
- Kiểu dữ liệu chủ yếu là `float64` (các biến số như giá, diện tích, thu nhập, rủi ro...) và `bool` (5 biến one-hot cho loại hình nhà: `type_condo`, `type_manufactured`, `type_multi`, `type_single`, `type_townhouse`).
- Không có cột dạng object/string — toàn bộ dữ liệu đã được số hóa, phù hợp để phân tích/tính toán trực tiếp.

In [ ]:
dtype_summary = df.dtypes.value_counts()
print("Phân bố kiểu dữ liệu:")
print(dtype_summary)


## 2. Thống kê mô tả (Descriptive Statistics)

Tính các chỉ số `mean`, `median`, `min`, `max`, `std` cho các biến số cốt lõi: `price`, `living`, `lot_sqft`, `income`.

In [ ]:
core_vars = ['price', 'living', 'lot_sqft', 'income']

desc_stats = df[core_vars].agg(['mean', 'median', 'min', 'max', 'std']).T
desc_stats.columns = ['Mean', 'Median', 'Min', 'Max', 'Std']
desc_stats


### Nhận diện hiện tượng lệch phải (skewness) của giá nhà

In [ ]:
price_mean = df['price'].mean()
price_median = df['price'].median()
price_skew = df['price'].skew()

gap = price_mean - price_median
gap_pct = gap / price_median * 100

print(f"Mean price   : {price_mean:,.0f}")
print(f"Median price : {price_median:,.0f}")
print(f"Chênh lệch (Mean - Median): {gap:,.0f} ({gap_pct:.1f}% so với Median)")
print(f"Hệ số bất đối xứng (skewness): {price_skew:.3f}")


**Nhận xét:**
- Mean của `price` lớn hơn đáng kể so với Median, và hệ số skewness dương (>1) cho thấy phân phối giá nhà **lệch phải (right-skewed)**.
- Điều này có nghĩa là phần lớn nhà có giá tập trung ở mức thấp/trung bình, nhưng tồn tại một số ít căn nhà có giá rất cao (outliers) kéo Mean lên cao hơn Median.
- Khi mô hình hóa hoặc trực quan hóa `price`, nên cân nhắc sử dụng phép biến đổi log hoặc xử lý outliers.

In [ ]:
# Kiểm tra skewness của tất cả các biến số cốt lõi
skew_table = df[core_vars].skew().sort_values(ascending=False).to_frame('Skewness')
skew_table


## 3. Phân tích dữ liệu khuyết (Missing Values) và trùng lặp (Duplicates)

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_table = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_table = missing_table[missing_table['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if missing_table.empty:
    print("Không có giá trị khuyết (missing values) trong bộ dữ liệu.")
else:
    print(missing_table)


In [ ]:
n_duplicates = df.duplicated().sum()
print(f"Số hàng trùng lặp: {n_duplicates}")

if n_duplicates > 0:
    df_clean = df.drop_duplicates().reset_index(drop=True)
    print(f"Đã loại bỏ {n_duplicates} hàng trùng lặp.")
    print(f"Số dòng sau khi làm sạch: {df_clean.shape[0]}")
else:
    df_clean = df.copy()
    print("Không có hàng trùng lặp. Dữ liệu đã sạch, giữ nguyên số dòng.")


**Nhận xét:** Bộ dữ liệu `zillow_final.csv` không có giá trị khuyết và không có hàng trùng lặp — đây là dữ liệu đã được làm sạch từ trước. `df_clean` được tạo ra để sử dụng thống nhất cho các bước phân tích tiếp theo (dù trong trường hợp này `df_clean` giống hệt `df`).

## 4. Bảng thống kê loại hình nhà và phân tích nhóm

Nhóm dữ liệu theo các biến one-hot `type_single`, `type_condo`, `type_multi`, `type_townhouse`, `type_manufactured` để tính số lượng và giá trung bình cho từng loại, từ đó xác định phân khúc đắt giá nhất.

In [ ]:
type_cols = ['type_single', 'type_condo', 'type_manufactured', 'type_multi', 'type_townhouse']

# Tạo 1 cột 'house_type' dạng nhãn duy nhất từ các cột one-hot để dễ groupby
def get_house_type(row):
    for col in type_cols:
        if row[col]:
            return col.replace('type_', '')
    return 'unknown'

df_clean['house_type'] = df_clean[type_cols].apply(get_house_type, axis=1)

df_clean['house_type'].value_counts()


In [ ]:
type_summary = df_clean.groupby('house_type')['price'].agg(
    count='count',
    mean_price='mean',
    median_price='median',
    min_price='min',
    max_price='max',
    std_price='std'
).sort_values('mean_price', ascending=False)

type_summary


In [ ]:
most_expensive_type = type_summary['mean_price'].idxmax()
most_expensive_value = type_summary['mean_price'].max()

print(f"Phân khúc nhà có giá trung bình cao nhất: '{most_expensive_type}'")
print(f"Giá trung bình: {most_expensive_value:,.0f}")


**Nhận xét:** Bảng trên cho thấy số lượng bất động sản và mức giá trung bình/median theo từng loại hình nhà. Loại hình có giá trung bình cao nhất được xác định phía trên là phân khúc đắt giá nhất trong bộ dữ liệu — thông tin này hữu ích để định hướng các phân tích hoặc mô hình dự đoán giá theo từng phân khúc riêng biệt.

In [ ]:
# Thêm góc nhìn bổ sung: diện tích living trung bình theo từng loại nhà
type_living_summary = df_clean.groupby('house_type')['living'].agg(
    count='count',
    mean_living='mean',
    median_living='median'
).sort_values('mean_living', ascending=False)

type_living_summary


## 5. Tính toán ma trận tương quan số (Correlation Matrix)

Trích xuất hệ số tương quan (Pearson) giữa tất cả các biến số với biến mục tiêu `price` để sàng lọc sơ bộ các đặc trưng quan trọng.

In [ ]:
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()

corr_matrix = df_clean[numeric_cols].corr()
corr_matrix.shape


In [ ]:
price_corr = corr_matrix['price'].drop('price').sort_values(key=lambda s: s.abs(), ascending=False)

price_corr_table = price_corr.to_frame('Correlation with price')
price_corr_table


In [ ]:
top_positive = price_corr.sort_values(ascending=False).head(5)
top_negative = price_corr.sort_values(ascending=True).head(5)

print("Top 5 biến có tương quan DƯƠNG mạnh nhất với price:")
print(top_positive)
print()
print("Top 5 biến có tương quan ÂM mạnh nhất với price:")
print(top_negative)


**Nhận xét sơ bộ:**
- Các biến thể hiện quy mô/diện tích nhà (như `living`) và các biến kinh tế - xã hội (như `income`, `bachelor`, `rent`) thường có tương quan dương đáng kể với `price`.
- Các biến rủi ro (`risk_*`) và khoảng cách (`dist_*`) có thể tương quan âm hoặc yếu, tùy đặc điểm khu vực.
- Danh sách hệ số tương quan phía trên là bước sàng lọc sơ bộ (feature screening) — các biến có |correlation| cao là ứng viên tiềm năng cho các bước phân tích/mô hình hóa tiếp theo. Cần lưu ý rằng tương quan không đồng nghĩa với quan hệ nhân quả.

In [ ]:
# Ma trận tương quan giữa price và các biến lõi hiện có trong dataset final.
key_features = ['price', 'living', 'lot_sqft', 'income', 'bed', 'bath', 'bachelor', 'rent']
corr_matrix.loc[key_features, key_features]


## 6. Tổng kết

- Dữ liệu gồm **4,468 dòng, 33 cột**, không có missing values và không có duplicates → chất lượng dữ liệu sạch.
- Biến mục tiêu `price` có phân phối **lệch phải** (Mean > Median), cần lưu ý khi mô hình hóa.
- Phân khúc nhà đắt giá nhất đã được xác định qua bước groupby theo `house_type`.
- Bảng tương quan với `price` cung cấp cơ sở để lựa chọn đặc trưng (feature selection) cho các bước phân tích/trực quan hóa hoặc mô hình hóa tiếp theo (ví dụ trong file `data_visualize.ipynb`).